### EXERCISE 5 FEATURE ENGINEER API BY ALENA(254) & RENATA(093)

In [13]:
texts = ['Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri', 'Cuti Bersama Idul Fitri',
         'Hari Idul Fitri', 'Cuti Bersama Idul Fitri', 'Idul Adha (Lebaran Haji)', 'Cuti Bersama Idul Fitri',
         'Idul Adha (Lebaran Haji)', 'Cuti Bersama Idul Fitri', 'Hari Idul Fitri', 'Cuti Bersama Idul Fitri',
         'Hari Idul Fitri', 'Cuti Bersama Idul Fitri', 'Hari Idul Fitri']

hasil_preprocess = {
    'Cuti Bersama Idul Fitri':  ['cuti', 'idul', 'fitri'],
    'Hari Idul Fitri':          ['idul', 'fitri'],
    'Idul Adha (Lebaran Haji)': ['idul', 'adha', 'lebaran', 'haji'],
}
preprocessed = [hasil_preprocess[t] for t in texts]

In [17]:
# df sudah ada dari Exercise 4
df['preprocessed'] = df['prepro'].apply(
    lambda tokens: ' '.join(tokens) if isinstance(tokens, list) else tokens
)
df_pre = df['preprocessed']
df_pre

0            cuti idul fitri
1            cuti idul fitri
2            cuti idul fitri
3            cuti idul fitri
4                 idul fitri
5            cuti idul fitri
6     idul adha lebaran haji
7            cuti idul fitri
8     idul adha lebaran haji
9            cuti idul fitri
10                idul fitri
11           cuti idul fitri
12                idul fitri
13           cuti idul fitri
14                idul fitri
Name: preprocessed, dtype: object

In [18]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(binary=True)
X = vectorizer.fit_transform(df_pre)
df_FE_ohe = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df_FE_ohe

,adha,cuti,fitri,haji,idul,lebaran
0,0,1,1,0,1,0
1,0,1,1,0,1,0
2,0,1,1,0,1,0
3,0,1,1,0,1,0
4,0,0,1,0,1,0
5,0,1,1,0,1,0
6,1,0,0,1,1,1
7,0,1,1,0,1,0
8,1,0,0,1,1,1
9,0,1,1,0,1,0


In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(df_pre)
df_FE_tfidf = pd.DataFrame(X.toarray(), columns=vectorizer.get_feature_names_out())
df_FE_tfidf

,adha,cuti,fitri,haji,idul,lebaran
0,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
1,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
2,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
3,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
4,0.000000,0.000000,0.749895,0.000000,0.661557,0.000000
5,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
6,0.564345,0.000000,0.000000,0.564345,0.211051,0.564345
7,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000
8,0.564345,0.000000,0.000000,0.564345,0.211051,0.564345
9,0.000000,0.697177,0.537599,0.000000,0.474269,0.000000


In [20]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk import word_tokenize
from tqdm import tqdm

sentences = [word_tokenize(t.lower()) for t in tqdm(df_pre)]
sentences

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\alena\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\alena\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
100%|██████████| 15/15 [00:00<00:00, 122.02it/s]


[['cuti', 'idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'adha', 'lebaran', 'haji'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'adha', 'lebaran', 'haji'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'fitri'],
 ['cuti', 'idul', 'fitri'],
 ['idul', 'fitri']]

In [21]:
import gensim.models

model = gensim.models.Word2Vec(sentences, min_count=1, vector_size=100, window=5, sg=0)
model.save('kalender_libur.w2v')
model = gensim.models.Word2Vec.load('kalender_libur.w2v')

w2v = model.wv
print(w2v.index_to_key)              # seluruh vocab (cuma 6 kata)
print(w2v.vectors.shape, w2v.vector_size)
print(w2v.similar_by_key('idul'))    # kata paling mirip dengan 'idul'

import numpy as np
def get_document_vector(tokens, model):
    valid_vectors = [model.wv[w] for w in tokens if w in model.wv]
    return np.mean(valid_vectors, axis=0)

X_features = np.array([get_document_vector(doc, model) for doc in sentences])
pd.DataFrame(X_features)

['idul', 'fitri', 'cuti', 'haji', 'lebaran', 'adha']
(6, 100) 100
[('fitri', -0.010839174501597881), ('lebaran', -0.02775035798549652), ('cuti', -0.051958419382572174), ('adha', -0.05987629294395447), ('haji', -0.1116705909371376)]


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
1,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
2,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
3,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
4,-0.004578,0.001951,0.005147,0.007376,-0.000919,-0.006643,0.003783,0.007511,-0.003928,-0.004969,...,0.001360,-0.000693,0.002835,-0.003832,0.003452,0.003862,-0.001786,-0.004717,-0.004304,0.005450
5,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
6,-0.006161,0.003227,-0.000786,-0.001130,-0.002602,-0.001698,0.003708,0.005430,-0.002390,-0.002731,...,0.001603,0.000838,-0.000062,-0.002398,0.007958,0.000415,0.000394,-0.004591,0.000536,-0.000705
7,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294
8,-0.006161,0.003227,-0.000786,-0.001130,-0.002602,-0.001698,0.003708,0.005430,-0.002390,-0.002731,...,0.001603,0.000838,-0.000062,-0.002398,0.007958,0.000415,0.000394,-0.004591,0.000536,-0.000705
9,-0.003021,0.002327,0.001161,0.004460,0.001943,-0.001980,0.001298,0.005889,-0.005392,-0.001244,...,-0.000597,0.001439,0.004951,-0.003921,0.004958,0.004368,0.000768,-0.002975,-0.000131,0.001294


In [28]:
!pip install gensim -q


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [30]:
model_fasttext = gensim.models.FastText(
    sentences,
    min_count=1,
    vector_size=100,
    window=5,
    min_n=3,
    max_n=6,
    epochs=10
)
model_fasttext.save('kalender_libur.fasttext')
model_fasttext = gensim.models.FastText.load('kalender_libur.fasttext')

fastext = model_fasttext.wv
print(fastext.index_to_key)
print(fastext.vectors.shape)
print(fastext.similar_by_key('idul'))

X_features_ft = np.array([get_document_vector(doc, model_fasttext) for doc in sentences])
pd.DataFrame(X_features_ft)

['idul', 'fitri', 'cuti', 'haji', 'lebaran', 'adha']
(6, 100)
[('fitri', 0.0855768471956253), ('cuti', 0.00916230771690607), ('haji', -0.030660511925816536), ('adha', -0.03496657311916351), ('lebaran', -0.08940248936414719)]


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
1,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
2,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
3,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
4,0.000434,0.002183,-0.000077,-0.000807,0.000228,-0.001485,-0.000781,0.001589,0.000957,-0.000179,...,0.000704,0.000695,-0.001405,-0.001452,0.000550,0.000566,-0.000483,0.000996,-0.000238,-0.000483
5,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
6,-0.000079,-0.000144,-0.001028,-0.001787,-0.000225,-0.000140,-0.000579,0.001721,0.000042,-0.001008,...,-0.000214,0.000196,-0.000654,-0.001313,0.000148,0.001278,-0.000180,-0.000994,-0.000649,0.000605
7,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719
8,-0.000079,-0.000144,-0.001028,-0.001787,-0.000225,-0.000140,-0.000579,0.001721,0.000042,-0.001008,...,-0.000214,0.000196,-0.000654,-0.001313,0.000148,0.001278,-0.000180,-0.000994,-0.000649,0.000605
9,-0.000189,0.001341,0.000021,0.000264,0.000508,-0.000356,-0.000528,0.000411,0.000064,0.000537,...,0.001647,0.000568,-0.001116,-0.001502,-0.000022,0.000610,-0.000278,0.001434,0.000241,-0.000719


In [31]:
!pip install umap-learn -q
from umap import UMAP
import plotly.express as px

X = UMAP(n_neighbors=3).fit_transform(w2v.vectors)   # n_neighbors dikecilkan karena vocab cuma 6
df2 = pd.DataFrame(X, columns=['umap1', 'umap2'])
df2['word'] = w2v.index_to_key

fig = px.scatter(df2, x='umap1', y='umap2', text='word')
fig.update_traces(textposition='top center')
fig.update_layout(height=600, title_text='Word2Vec Visualization')
fig.show()


[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip
